# Week 1: Stability & Reproducibility Experiments**Purpose**: Validate GAT 21d IC=0.04420 reproducibility across random seeds.## Experiments1. **Multi-seed GAT 21d** (5 seeds) — IC mean±std2. **Multi-seed LightGBM 21d** (control) — evaluation pipeline stability3. **Training diagnostics** — per-epoch val IC curves4. **LSTM Baseline** — sequence model without graph structure## InfrastructureCells 1-10 are identical to `v3_ranking_pipeline.ipynb`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SETUP — imports, environment, reproducibility
# ══════════════════════════════════════════════════════════════════

import os, sys, time, random, warnings, gc
from collections import defaultdict
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from scipy.sparse import csr_matrix
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Environment ---
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/GNN\u6d4b\u8bd5'
    os.chdir(DRIVE_FOLDER)
    print(f'Colab: working dir = {DRIVE_FOLDER}')
    !pip install --quiet torch_geometric lightgbm
except ImportError:
    os.chdir('/Users/heruixi/Desktop/GNN-Testing')
    print(f'Local: working dir = {os.getcwd()}')

for d in ['data/fullscale', 'data/reference', 'plots', 'experiments']:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS — All tunable values in one place
# ══════════════════════════════════════════════════════════════════
PARAMS = {
    # --- Data paths ---
    'events_file':  'data/fullscale/sp500_news_events.parquet',
    'emb_file':     'data/fullscale/sp500_news_emb_finbert.npy',
    'sent_file':    'data/fullscale/sp500_news_sentiment_finbert.npy',
    'prices_file':  'data/reference/sp500_5y_prices.csv',
    'sectors_file': 'data/reference/sp500_sectors.csv',

    # --- Dynamic graph ---
    'corr_window':    126,    # trading days for rolling correlation
    'corr_threshold': 0.6,    # |Pearson r| edge inclusion
    'corr_step':      21,     # monthly re-estimation step

    # --- Prediction ---
    'horizons': [1, 5, 10, 21, 42, 63],  # forward return horizons (trading days)
    'default_horizon': 5,                 # primary horizon (MASTER default)

    # --- Time split ---
    'train_start': '2021-07-01',
    'train_end':   '2023-12-31',
    'val_end':     '2024-06-30',
    # test = everything after val_end

    # --- HGT architecture ---
    'hidden_channels': 64,
    'num_heads':       4,
    'num_hgt_layers':  2,
    'dropout':         0.3,

    # --- Training ---
    'lr':            1e-3,
    'weight_decay':  1e-4,
    'epochs':        100,
    'patience':      15,
    'grad_accum':    32,     # accumulate over N days before optimizer step

    # --- Portfolio evaluation ---
    'top_k':             30,   # long/short top-k stocks
    'transaction_cost':  15,   # bps round-trip

    # --- SelectiveNet ---
    'target_coverage': 0.2,
    'selection_lambda': 32.0,
}

print('Parameters:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# LOAD ALL RAW DATA
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# --- Prices: (num_days, num_tickers) wide format ---
prices = pd.read_csv(PARAMS['prices_file'], index_col=0, parse_dates=True)
prices.index.name = 'Date'
print(f'Prices: {prices.shape}  ({prices.index[0].date()} -> {prices.index[-1].date()})')

# --- Sectors ---
sector_df = pd.read_csv(PARAMS['sectors_file'])
sector_col = [c for c in sector_df.columns if 'sector' in c.lower()][0]
ticker_col = [c for c in sector_df.columns if c != sector_col][0]
sector_map = dict(zip(sector_df[ticker_col], sector_df[sector_col]))
unique_sectors = sorted(set(sector_map.values()))
print(f'Sectors: {len(unique_sectors)} unique')

# --- Events + Embeddings + Sentiment ---
events = pd.read_parquet(PARAMS['events_file'])
events['date'] = pd.to_datetime(events['date'], utc=True).dt.tz_localize(None).dt.normalize()
emb_all = np.load(PARAMS['emb_file'])   # fully load for speed
sent_all = np.load(PARAMS['sent_file'])
print(f'Events: {len(events):,}  Emb: {emb_all.shape}  Sent: {sent_all.shape}')

# --- Canonical ticker list (intersection of all sources) ---
price_tickers = set(prices.columns)
event_tickers = set(events['ticker'].unique())
sector_tickers = set(sector_map.keys())
valid_tickers = sorted(price_tickers & event_tickers & sector_tickers)
ticker_to_id = {t: i for i, t in enumerate(valid_tickers)}
num_stocks = len(valid_tickers)
print(f'Valid tickers: {num_stocks}')

# Filter prices to valid tickers
prices = prices[valid_tickers]
returns = prices.pct_change()
returns.iloc[0] = 0

# Trading day index
all_dates = prices.index
num_days = len(all_dates)
date_to_idx = {d: i for i, d in enumerate(all_dates)}
print(f'Trading days: {num_days}  ({all_dates[0].date()} -> {all_dates[-1].date()})')

print(f'\nAll data loaded in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1a: PRICE FEATURES — 9-dim per stock-day (strictly T-1 close)
# 3 windows (5/10/21d) x 3 stats (return mean, volatility, momentum)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

price_feat_dict = {}
for w in [5, 10, 21]:
    # Mean daily return over past w days (ending T-1)
    price_feat_dict[f'ret_mean_{w}d'] = returns.rolling(w).mean().shift(1)
    # Volatility: std of daily returns (ending T-1)
    price_feat_dict[f'ret_std_{w}d'] = returns.rolling(w).std().shift(1)
    # Momentum: cumulative return = close[T-1] / close[T-1-w] - 1
    price_feat_dict[f'momentum_{w}d'] = (prices.shift(1) / prices.shift(1 + w) - 1)

price_feat_names = list(price_feat_dict.keys())

# Stack: (num_days, num_stocks, 9)
price_features = np.stack(
    [price_feat_dict[name].values for name in price_feat_names], axis=-1
).astype(np.float32)

nan_before = np.isnan(price_features).sum()
price_features = np.nan_to_num(price_features, 0.0)
print(f'Price features: {price_features.shape}')
print(f'  Features: {price_feat_names}')
print(f'  NaN filled: {nan_before:,} -> 0 (first ~21 days lack lookback)')

del price_feat_dict
gc.collect()
print(f'N1a complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1b: NEWS FEATURES -> CALENDAR GRID
# For each (trading_day, ticker): mean-pooled FinBERT 768d + 3d sentiment
# Days without news -> zero vector + has_news=0
# Also precompute per-day event metadata for graph construction (N2)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# Map events to calendar grid
events['event_idx'] = np.arange(len(events))
events_v = events[events['ticker'].isin(ticker_to_id)].copy()
events_v['stock_id'] = events_v['ticker'].map(ticker_to_id)
events_v['day_idx'] = events_v['date'].map(date_to_idx)
events_v = events_v.dropna(subset=['day_idx'])
events_v['day_idx'] = events_v['day_idx'].astype(int)
print(f'Events mapped to grid: {len(events_v):,} / {len(events):,}')

# --- Mean-pool via sparse aggregation (fast) ---
group_keys = events_v['day_idx'].values * num_stocks + events_v['stock_id'].values
unique_keys, inverse = np.unique(group_keys, return_inverse=True)
group_counts = np.bincount(inverse).astype(np.float32)

# Sparse aggregation matrix: (num_groups, num_events_valid)
rows = inverse
cols = np.arange(len(events_v))
vals = 1.0 / group_counts[inverse]
agg_mat = csr_matrix((vals, (rows, cols)), shape=(len(unique_keys), len(events_v)))

# Aggregate embeddings and sentiment
event_idxs = events_v['event_idx'].values
agg_emb = (agg_mat @ emb_all[event_idxs]).astype(np.float32)   # (groups, 768)
agg_sent = (agg_mat @ sent_all[event_idxs]).astype(np.float32)  # (groups, 3)

# Scatter into (num_days, num_stocks, dim) arrays
news_emb = np.zeros((num_days, num_stocks, 768), dtype=np.float32)
news_sent = np.zeros((num_days, num_stocks, 3), dtype=np.float32)
has_news = np.zeros((num_days, num_stocks, 1), dtype=np.float32)

for i, gk in enumerate(unique_keys):
    d_idx = int(gk // num_stocks)
    s_idx = int(gk % num_stocks)
    news_emb[d_idx, s_idx] = agg_emb[i]
    news_sent[d_idx, s_idx] = agg_sent[i]
    has_news[d_idx, s_idx] = 1.0

news_coverage = has_news.sum() / (num_days * num_stocks) * 100
print(f'News coverage: {has_news.sum():.0f} / {num_days * num_stocks:,} stock-days '
      f'({news_coverage:.1f}%)')

# --- Combine all features: (num_days, num_stocks, 781) ---
# Layout: [price(9) | emb(768) | sent(3) | has_news(1)]
stock_features_np = np.concatenate(
    [price_features, news_emb, news_sent, has_news], axis=-1
)
print(f'Stock features: {stock_features_np.shape}  '
      f'({stock_features_np.nbytes / 1e9:.2f} GB)')

# --- Precompute per-day event data for graph construction (N2) ---
# Store only indices to save memory; load embeddings lazily
daily_events = {}
events_v_sorted = events_v.sort_values(['day_idx', 'stock_id'])
for day_idx, group in events_v_sorted.groupby('day_idx'):
    daily_events[int(day_idx)] = {
        'event_idxs': group['event_idx'].values,
        'stock_ids': group['stock_id'].values,
        'titles': group['title'].values if 'title' in group.columns else None,
    }

print(f'Daily event groups: {len(daily_events)} days')

del price_features, news_emb, news_sent, has_news, agg_emb, agg_sent, agg_mat
gc.collect()
print(f'\nN1b complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1c: MULTI-HORIZON LABELS (6 horizons)
# Label = Z-score of cross-sectional excess return
# Excess = stock_ret - equal_weight_market_ret
# Z-score = per-day normalization across stocks
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

labels = {}  # horizon -> (num_days, num_stocks) np array
label_valid = {}  # horizon -> (num_days, num_stocks) bool mask

for h in PARAMS['horizons']:
    # Forward return: (close[T+h] - close[T]) / close[T]
    fwd_ret = prices.shift(-h) / prices - 1  # (num_days, num_stocks)

    # Equal-weight market return per day
    market_ret = fwd_ret.mean(axis=1)  # (num_days,)

    # Excess return
    excess = fwd_ret.sub(market_ret, axis=0)  # (num_days, num_stocks)

    # Cross-sectional Z-score per day
    day_mean = excess.mean(axis=1)
    day_std = excess.std(axis=1)
    day_std[day_std < 1e-8] = 1.0  # avoid division by zero
    z_score = excess.sub(day_mean, axis=0).div(day_std, axis=0)

    # Valid mask: not NaN (last h days have NaN forward returns)
    valid = ~z_score.isna()

    labels[h] = z_score.values.astype(np.float32)
    labels[h] = np.nan_to_num(labels[h], 0.0)
    label_valid[h] = valid.values

    n_valid = valid.sum().sum()
    print(f'Horizon {h:2d}d: {n_valid:,} valid labels '
          f'(last {h} days = NaN, z-score mean={labels[h][valid.values].mean():.4f} '
          f'std={labels[h][valid.values].std():.4f})')

print(f'\nN1c complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1d: TIME SPLIT + DATA SUMMARY
# Train: 2021-07 -> 2023-12 | Val: 2024-01 -> 2024-06 | Test: 2024-07+
# ══════════════════════════════════════════════════════════════════

train_start = pd.Timestamp(PARAMS['train_start'])
train_end = pd.Timestamp(PARAMS['train_end'])
val_end = pd.Timestamp(PARAMS['val_end'])

train_mask = (all_dates >= train_start) & (all_dates <= train_end)
val_mask = (all_dates > train_end) & (all_dates <= val_end)
test_mask = (all_dates > val_end)

train_days = np.where(train_mask)[0]
val_days = np.where(val_mask)[0]
test_days = np.where(test_mask)[0]

print(f'=== Time Split ===')
print(f'Train: {all_dates[train_days[0]].date()} -> {all_dates[train_days[-1]].date()} '
      f'({len(train_days)} days)')
print(f'Val:   {all_dates[val_days[0]].date()} -> {all_dates[val_days[-1]].date()} '
      f'({len(val_days)} days)')
print(f'Test:  {all_dates[test_days[0]].date()} -> {all_dates[test_days[-1]].date()} '
      f'({len(test_days)} days)')

# Summary: stock-days with news per split
for name, days in [('Train', train_days), ('Val', val_days), ('Test', test_days)]:
    total = len(days) * num_stocks
    with_news = stock_features_np[days, :, -1].sum()  # has_news flag is last dim
    print(f'  {name}: {total:,} stock-days, {int(with_news):,} with news '
          f'({with_news/total*100:.1f}%)')

# Convert features to tensors
stock_features_t = torch.tensor(stock_features_np, dtype=torch.float32)
labels_t = {h: torch.tensor(labels[h], dtype=torch.float32) for h in PARAMS['horizons']}
label_valid_t = {h: torch.tensor(label_valid[h], dtype=torch.bool) for h in PARAMS['horizons']}

print(f'\nFeature tensor: {stock_features_t.shape}  ({stock_features_t.element_size() * stock_features_t.nelement() / 1e9:.2f} GB)')
print(f'Label tensors: {len(labels_t)} horizons')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N2: GRAPH CONSTRUCTION — 4 edge types
# 1. stock-stock (correlation): monthly dynamic, |r|>0.6, w=126
# 2. stock-stock (sector): static, same GICS sector
# 3. news->stock (mentions): daily, per event
# 4. stock-stock (co-occurrence): daily, same-article tickers
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# ------------------------------------------------------------------
# Edge Type 1: Correlation (monthly dynamic)
# ------------------------------------------------------------------
corr_w = PARAMS['corr_window']
corr_t = PARAMS['corr_threshold']
corr_step = PARAMS['corr_step']

# Build correlation edge snapshots
corr_snapshots = {}  # snapshot_idx -> edge_index tensor
corr_day_to_snapshot = {}  # day_idx -> snapshot_idx

snapshot_points = list(range(corr_w, num_days, corr_step))
print(f'Correlation snapshots: {len(snapshot_points)} (w={corr_w}, t={corr_t}, step={corr_step})')

for snap_idx, t_end in enumerate(snapshot_points):
    window_ret = returns.iloc[t_end - corr_w : t_end].values  # (w, num_stocks)
    corr_mat = np.corrcoef(window_ret.T)  # (num_stocks, num_stocks)
    np.fill_diagonal(corr_mat, 0)
    mask = np.abs(corr_mat) > corr_t
    src, dst = np.where(mask)
    corr_snapshots[snap_idx] = torch.tensor(
        np.stack([src, dst]), dtype=torch.long
    )

# Map each day to its most recent correlation snapshot
snap_idx = 0
for day_idx in range(num_days):
    # Find latest snapshot that ends at or before this day
    while snap_idx + 1 < len(snapshot_points) and snapshot_points[snap_idx + 1] <= day_idx:
        snap_idx += 1
    if snapshot_points[snap_idx] <= day_idx:
        corr_day_to_snapshot[day_idx] = snap_idx
    # Days before first snapshot use snapshot 0
    elif snap_idx == 0:
        corr_day_to_snapshot[day_idx] = 0

# Stats
for i in [0, len(snapshot_points)//2, len(snapshot_points)-1]:
    e = corr_snapshots[i]
    print(f'  Snapshot {i}: {e.shape[1]} edges, '
          f'density={e.shape[1]/(num_stocks*(num_stocks-1))*100:.1f}%')

# ------------------------------------------------------------------
# Edge Type 2: Sector (static)
# ------------------------------------------------------------------
sector_edges_src, sector_edges_dst = [], []
sector_groups = defaultdict(list)
for t_name in valid_tickers:
    if t_name in sector_map:
        sector_groups[sector_map[t_name]].append(ticker_to_id[t_name])

for sector, members in sector_groups.items():
    for i in range(len(members)):
        for j in range(len(members)):
            if i != j:
                sector_edges_src.append(members[i])
                sector_edges_dst.append(members[j])

sector_edge_index = torch.tensor(
    [sector_edges_src, sector_edges_dst], dtype=torch.long
)
print(f'Sector edges: {sector_edge_index.shape[1]:,} '
      f'({len(sector_groups)} sectors)')

# ------------------------------------------------------------------
# Edge Types 3 & 4: News mention + co-occurrence (daily)
# ------------------------------------------------------------------
daily_graph_data = {}  # day_idx -> dict with mention_edges, cooccur_edges, news_feat

for day_idx in range(num_days):
    if day_idx not in daily_events:
        daily_graph_data[day_idx] = {
            'n_news': 0,
            'news_feat': torch.zeros(1, 771, dtype=torch.float32),  # dummy node
            'mention_edges': torch.zeros(2, 0, dtype=torch.long),
            'cooccur_edges': torch.zeros(2, 0, dtype=torch.long),
        }
        continue

    de = daily_events[day_idx]
    ev_idxs = de['event_idxs']
    stock_ids = de['stock_ids']
    n_events = len(ev_idxs)

    # News node features: FinBERT embedding + sentiment = 771-dim
    feat = np.concatenate([emb_all[ev_idxs], sent_all[ev_idxs]], axis=1)  # (N, 771)

    # Mention edges: news_node_i -> stock_id
    mention_edges = torch.tensor(
        np.stack([np.arange(n_events), stock_ids]), dtype=torch.long
    )

    # Co-occurrence: stocks cited in same article (group by title)
    cooccur_src, cooccur_dst = [], []
    if de['titles'] is not None:
        title_to_stocks = defaultdict(set)
        for title, sid in zip(de['titles'], stock_ids):
            title_to_stocks[title].add(int(sid))
        for title, sids in title_to_stocks.items():
            sids = list(sids)
            if len(sids) >= 2:
                for s1, s2 in combinations(sids, 2):
                    cooccur_src.extend([s1, s2])
                    cooccur_dst.extend([s2, s1])

    if cooccur_src:
        cooccur_edges = torch.tensor([cooccur_src, cooccur_dst], dtype=torch.long)
    else:
        cooccur_edges = torch.zeros(2, 0, dtype=torch.long)

    daily_graph_data[day_idx] = {
        'n_news': n_events,
        'news_feat': torch.tensor(feat, dtype=torch.float32),
        'mention_edges': mention_edges,
        'cooccur_edges': cooccur_edges,
    }

# Stats
n_mention_total = sum(d['mention_edges'].shape[1] for d in daily_graph_data.values())
n_cooccur_total = sum(d['cooccur_edges'].shape[1] for d in daily_graph_data.values())
print(f'News mention edges: {n_mention_total:,} total '
      f'({n_mention_total/num_days:.0f}/day avg)')
print(f'Co-occurrence edges: {n_cooccur_total:,} total '
      f'({n_cooccur_total/num_days:.0f}/day avg)')

# ------------------------------------------------------------------
# HeteroData metadata
# ------------------------------------------------------------------
METADATA = (
    ['stock', 'news'],
    [
        ('stock', 'corr', 'stock'),
        ('stock', 'sector', 'stock'),
        ('news', 'mentions', 'stock'),
        ('stock', 'cooccur', 'stock'),
    ]
)
# ------------------------------------------------------------------
# N2d: Jaccard Audit — correlation edge dynamics across snapshots
# ------------------------------------------------------------------
jaccard_values = []
jaccard_dates = []
for i in range(1, len(snapshot_points)):
    s1 = set(map(tuple, corr_snapshots[i-1].t().numpy().tolist()))
    s2 = set(map(tuple, corr_snapshots[i].t().numpy().tolist()))
    union = len(s1 | s2)
    jaccard = len(s1 & s2) / union if union > 0 else 1.0
    jaccard_values.append(jaccard)
    jaccard_dates.append(all_dates[snapshot_points[i]])

print(f'\nJaccard similarity (adjacent snapshots):')
print(f'  Mean: {np.mean(jaccard_values):.3f}  Std: {np.std(jaccard_values):.3f}')
print(f'  Min:  {np.min(jaccard_values):.3f} at {jaccard_dates[np.argmin(jaccard_values)].date()}')

print(f'\nGraph metadata:')
print(f'  Node types: {METADATA[0]}')
print(f'  Edge types: {[e[1] for e in METADATA[1]]}')
print(f'\nN2 complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# HELPER: Build HeteroData for a single day
# ══════════════════════════════════════════════════════════════════
from torch_geometric.data import HeteroData

def build_hetero_data(day_idx, to_device=None):
    """Construct HeteroData graph for a single trading day."""
    dgd = daily_graph_data[day_idx]

    data = HeteroData()

    # --- Stock node features ---
    data['stock'].x = stock_features_t[day_idx]  # (num_stocks, 781)
    data['stock'].num_nodes = num_stocks

    # --- News node features ---
    if dgd['n_news'] > 0:
        data['news'].x = dgd['news_feat']
        data['news'].num_nodes = dgd['n_news']
    else:
        # Dummy news node (ensures metadata consistency)
        data['news'].x = dgd['news_feat']  # (1, 771) zeros
        data['news'].num_nodes = 1

    # --- Edge type 1: Correlation ---
    snap_idx = corr_day_to_snapshot.get(day_idx, 0)
    data['stock', 'corr', 'stock'].edge_index = corr_snapshots[snap_idx]

    # --- Edge type 2: Sector (static) ---
    data['stock', 'sector', 'stock'].edge_index = sector_edge_index

    # --- Edge type 3: News mentions ---
    if dgd['n_news'] > 0:
        data['news', 'mentions', 'stock'].edge_index = dgd['mention_edges']
    else:
        data['news', 'mentions', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)

    # --- Edge type 4: Co-occurrence ---
    data['stock', 'cooccur', 'stock'].edge_index = dgd['cooccur_edges']

    if to_device is not None:
        data = data.to(to_device)
    return data

# Validation: build sample graph
sample_data = build_hetero_data(train_days[len(train_days)//2])
print('=== Sample HeteroData ===')
print(sample_data)
for et in METADATA[1]:
    ei = sample_data[et].edge_index
    print(f'  {et[1]}: {ei.shape[1]} edges')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVALUATION UTILITIES — IC, ICIR, Portfolio Backtest
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

def compute_daily_ic(predictions, actuals, valid_mask, day_indices):
    """
    Compute daily IC (Spearman correlation) cross-sectionally.

    Parameters
    ----------
    predictions : np.ndarray, shape (num_all_days, num_stocks)
    actuals : np.ndarray, shape (num_all_days, num_stocks)
    valid_mask : np.ndarray, shape (num_all_days, num_stocks)
    day_indices : array of day indices to evaluate

    Returns
    -------
    ic_series : np.ndarray of daily IC values
    dates : corresponding dates
    """
    ic_list = []
    valid_dates = []
    for d in day_indices:
        mask = valid_mask[d]
        pred = predictions[d][mask]
        actual = actuals[d][mask]
        if len(pred) >= 30:
            ic, _ = spearmanr(pred, actual)
            if not np.isnan(ic):
                ic_list.append(ic)
                valid_dates.append(all_dates[d])
    return np.array(ic_list), valid_dates


def compute_metrics(predictions, horizon, day_indices, name=''):
    """Compute full ranking metrics for a model's predictions."""
    actuals = labels[horizon]
    valid = label_valid[horizon]

    ic_arr, ic_dates = compute_daily_ic(predictions, actuals, valid, day_indices)

    if len(ic_arr) == 0:
        return {'name': name, 'IC': 0, 'ICIR': 0, 'Rank_IC': 0}

    ic_mean = ic_arr.mean()
    ic_std = ic_arr.std()
    icir = ic_mean / ic_std if ic_std > 1e-8 else 0

    # --- Portfolio backtest ---
    top_k = PARAMS['top_k']
    tc_bps = PARAMS['transaction_cost']
    daily_long_ret = []
    daily_short_ret = []
    daily_ls_ret = []

    fwd_raw = prices.pct_change(horizon).shift(-horizon).values  # raw returns

    for d in day_indices:
        mask = valid[d]
        if mask.sum() < 2 * top_k:
            continue
        pred = predictions[d].copy()
        pred[~mask] = np.nan

        # Rank stocks
        valid_idx = np.where(mask)[0]
        scores = pred[valid_idx]
        ranks = np.argsort(np.argsort(-scores))  # 0 = best

        top_idx = valid_idx[ranks < top_k]
        bot_idx = valid_idx[ranks >= len(scores) - top_k]

        # Raw returns for this horizon
        ret_day = fwd_raw[d]
        long_ret = np.nanmean(ret_day[top_idx])
        short_ret = np.nanmean(ret_day[bot_idx])
        ls_ret = long_ret - short_ret

        daily_long_ret.append(long_ret)
        daily_short_ret.append(short_ret)
        daily_ls_ret.append(ls_ret)

    daily_ls_ret = np.array(daily_ls_ret)
    daily_long_ret = np.array(daily_long_ret)

    # Annualize (scale by 252/horizon for daily-equivalent)
    scale = 252 / horizon
    ann_ls = daily_ls_ret.mean() * scale
    ann_ls_vol = daily_ls_ret.std() * np.sqrt(scale) if len(daily_ls_ret) > 1 else 1e-8
    sharpe_ls = ann_ls / ann_ls_vol if ann_ls_vol > 1e-8 else 0

    ann_long = daily_long_ret.mean() * scale
    ann_long_vol = daily_long_ret.std() * np.sqrt(scale) if len(daily_long_ret) > 1 else 1e-8
    sharpe_long = ann_long / ann_long_vol if ann_long_vol > 1e-8 else 0

    # Transaction costs (rebalance each period)
    n_rebal = len(daily_ls_ret)
    tc_total = n_rebal * (tc_bps / 10000) * 2  # 2x for long+short turnover (approx)
    ann_ls_net = ann_ls - tc_total / max(1, n_rebal / scale)

    # Max drawdown
    cum = np.cumsum(daily_ls_ret)
    peak = np.maximum.accumulate(cum)
    max_dd = (peak - cum).max() if len(cum) > 0 else 0

    return {
        'name': name,
        'IC': round(ic_mean, 5),
        'ICIR': round(icir, 3),
        'IC_std': round(ic_std, 5),
        'Sharpe_LS': round(sharpe_ls, 3),
        'Sharpe_Long': round(sharpe_long, 3),
        'Ann_LS': f'{ann_ls*100:.2f}%',
        'Ann_LS_net': f'{ann_ls_net*100:.2f}%',
        'MaxDD': f'{max_dd*100:.2f}%',
        'n_days': len(ic_arr),
    }


def print_results_table(results_list, title=''):
    """Pretty-print a list of metric dicts."""
    if title:
        print(f'\n=== {title} ===')
    df = pd.DataFrame(results_list)
    display_cols = ['name', 'IC', 'ICIR', 'Sharpe_LS', 'Sharpe_Long',
                    'Ann_LS', 'Ann_LS_net', 'MaxDD', 'n_days']
    cols = [c for c in display_cols if c in df.columns]
    print(df[cols].to_string(index=False))
    return df

print('Evaluation utilities loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# LIGHTGBM HELPER — for multi-seed comparison
# ══════════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('LightGBM not available')


def train_lightgbm(horizon, train_days, val_days, test_days, seed=42, name='LightGBM'):
    """Train LightGBM for a given horizon and return metrics."""
    if not HAS_LGB:
        return None

    h = horizon

    # Flatten data
    def flatten(day_indices):
        X_list, y_list = [], []
        for d in day_indices:
            mask = label_valid[h][d]
            X_list.append(stock_features_np[d][mask])
            y_list.append(labels[h][d][mask])
        return np.vstack(X_list), np.concatenate(y_list)

    X_train, y_train = flatten(train_days)
    X_val, y_val = flatten(val_days)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    model = lgb.LGBMRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=seed, n_jobs=-1, verbose=-1,
    )
    model.fit(
        X_train_s, y_train,
        eval_set=[(X_val_s, y_val)],
        callbacks=[lgb.early_stopping(20, verbose=False)],
    )

    # Predict per-day
    pred = np.zeros((num_days, num_stocks), dtype=np.float32)
    for d in range(num_days):
        x = scaler.transform(stock_features_np[d])
        pred[d] = model.predict(x)

    return compute_metrics(pred, h, test_days, name)


print('LightGBM helper loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N3b: HGT MODEL DEFINITION
# Heterogeneous Graph Transformer for cross-sectional ranking
# ══════════════════════════════════════════════════════════════════
from torch_geometric.nn import HGTConv

class RankingHGT(nn.Module):
    """
    Heterogeneous Graph Transformer for stock ranking prediction.

    Node types: stock (781-dim), news (771-dim)
    Edge types: corr, sector, mentions, cooccur
    Output: per-stock ranking score (scalar)
    """
    def __init__(self, stock_in, news_in, hidden, metadata,
                 num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.stock_lin = nn.Linear(stock_in, hidden)
        self.news_lin = nn.Linear(news_in, hidden)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(HGTConv(hidden, hidden, metadata, heads=num_heads))
            self.norms.append(nn.LayerNorm(hidden))

        self.dropout = dropout
        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        # Project to hidden space
        h_dict = {
            'stock': F.relu(self.stock_lin(x_dict['stock'])),
        }
        if 'news' in x_dict and x_dict['news'].shape[0] > 0:
            h_dict['news'] = F.relu(self.news_lin(x_dict['news']))
        else:
            h_dict['news'] = torch.zeros(1, self.stock_lin.out_features,
                                          device=x_dict['stock'].device)

        # HGT message passing with residual + LayerNorm
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h_dict, edge_index_dict)
            # Residual + norm for stock nodes
            h_dict['stock'] = norm(
                F.dropout(h_new['stock'], p=self.dropout, training=self.training)
                + h_dict['stock']
            )
            # News nodes: keep original (no incoming edges update them)
            if 'news' in h_new:
                h_dict['news'] = h_new['news']

        # Ranking score for each stock
        return self.ranking_head(h_dict['stock']).squeeze(-1)

    def get_stock_embeddings(self, x_dict, edge_index_dict):
        """Return stock node embeddings before ranking head (for SelectiveNet)."""
        h_dict = {
            'stock': F.relu(self.stock_lin(x_dict['stock'])),
        }
        if 'news' in x_dict and x_dict['news'].shape[0] > 0:
            h_dict['news'] = F.relu(self.news_lin(x_dict['news']))
        else:
            h_dict['news'] = torch.zeros(1, self.stock_lin.out_features,
                                          device=x_dict['stock'].device)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h_dict, edge_index_dict)
            h_dict['stock'] = norm(
                F.dropout(h_new['stock'], p=self.dropout, training=self.training)
                + h_dict['stock']
            )
            if 'news' in h_new:
                h_dict['news'] = h_new['news']
        return h_dict['stock']


# --- Also define simpler GNN baselines for ablation ---

class RankingGNN(nn.Module):
    """Homogeneous GNN (GraphSAGE or GAT) for ablation against HGT.
    Uses only stock nodes and correlation+sector edges."""
    def __init__(self, in_channels, hidden, conv_type='sage', num_layers=2, dropout=0.3):
        super().__init__()
        self.lin = nn.Linear(in_channels, hidden)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            if conv_type == 'sage':
                from torch_geometric.nn import SAGEConv
                self.convs.append(SAGEConv(hidden, hidden))
            elif conv_type == 'gat':
                from torch_geometric.nn import GATConv
                self.convs.append(GATConv(hidden, hidden // 4, heads=4, concat=True))
            self.norms.append(nn.LayerNorm(hidden))

        self.dropout = dropout
        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index):
        h = F.relu(self.lin(x))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h = norm(F.dropout(h_new, p=self.dropout, training=self.training) + h)
        return self.ranking_head(h).squeeze(-1)

    def get_stock_embeddings(self, x, edge_index):
        """Return stock node embeddings before ranking head (for SelectiveNet)."""
        h = F.relu(self.lin(x))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h = norm(F.dropout(h_new, p=self.dropout, training=self.training) + h)
        return h


print('Models defined: RankingHGT, RankingGNN')
print(f'  HGT: stock_in=781, news_in=771, hidden={PARAMS["hidden_channels"]}, '
      f'heads={PARAMS["num_heads"]}, layers={PARAMS["num_hgt_layers"]}')

# ══════════════════════════════════════════════════════════════════
# LSTM BASELINE �� sequence model without graph structure
# Uses past SEQ_LEN days of features per stock as input sequence
# ══════════════════════════════════════════════════════════════════

class RankingLSTM(nn.Module):
    """LSTM baseline: processes per-stock time series without graph structure.
    Input: (batch_stocks, seq_len, feature_dim) -> (batch_stocks,) ranking scores."""

    def __init__(self, in_channels=781, hidden=64, num_layers=1, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_channels, hidden)
        self.lstm = nn.LSTM(hidden, hidden, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_seq):
        """
        Args:
            x_seq: (num_stocks, seq_len, in_channels) — feature sequence per stock
        Returns:
            scores: (num_stocks,) ranking scores
        """
        h = F.relu(self.proj(x_seq))  # (stocks, seq_len, hidden)
        _, (h_n, _) = self.lstm(h)     # h_n: (layers, stocks, hidden)
        out = h_n[-1]                   # (stocks, hidden) — last layer's hidden
        return self.ranking_head(out).squeeze(-1)


print('RankingLSTM defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ENHANCED TRAINING LOOP — per-epoch val IC tracking + seed parameter
# ══════════════════════════════════════════════════════════════════

def set_seed(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_gat_with_diagnostics(horizon, train_days, val_days, test_days,
                                edge_types=['corr', 'sector'],
                                seed=42, epochs=None, patience=None,
                                grad_accum=None, verbose=True):
    """
    Train GAT model with full diagnostics.
    Returns: (metrics_dict, history_dict, predictions_array)

    history includes per-epoch: train_loss, val_loss, val_ic
    """
    epochs = epochs or PARAMS['epochs']
    patience = patience or PARAMS['patience']
    grad_accum = grad_accum or PARAMS['grad_accum']

    # Set seed before model creation
    set_seed(seed)

    model = RankingGNN(
        in_channels=stock_features_t.shape[-1],
        hidden=PARAMS['hidden_channels'],
        conv_type='gat',
        num_layers=PARAMS['num_hgt_layers'],
        dropout=PARAMS['dropout'],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS['lr'],
                                 weight_decay=PARAMS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_ic': [], 'lr': []}

    lab = labels_t[horizon]
    valid = label_valid_t[horizon]

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss_sum = 0
        train_count = 0
        optimizer.zero_grad()

        day_order = train_days[np.random.permutation(len(train_days))]
        for step, d in enumerate(day_order):
            x = stock_features_t[d].to(device)

            # Combine edge types
            edge_lists = []
            if 'corr' in edge_types:
                snap = corr_day_to_snapshot.get(d, 0)
                edge_lists.append(corr_snapshots[snap])
            if 'sector' in edge_types:
                edge_lists.append(sector_edge_index)
            edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)

            pred = model(x, edge_index)
            mask = valid[d].to(device)
            target = lab[d].to(device)
            if mask.sum() < 10:
                continue
            loss = F.mse_loss(pred[mask], target[mask]) / grad_accum
            loss.backward()
            train_loss_sum += loss.item() * grad_accum
            train_count += 1
            if (step + 1) % grad_accum == 0 or step == len(day_order) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        avg_train = train_loss_sum / max(train_count, 1)
        history['train_loss'].append(avg_train)

        # --- Validate (compute both loss and IC) ---
        model.eval()
        val_loss_sum = 0
        val_count = 0
        val_preds = np.zeros((num_days, num_stocks), dtype=np.float32)

        with torch.no_grad():
            for d in val_days:
                x = stock_features_t[d].to(device)
                edge_lists = []
                if 'corr' in edge_types:
                    snap = corr_day_to_snapshot.get(d, 0)
                    edge_lists.append(corr_snapshots[snap])
                if 'sector' in edge_types:
                    edge_lists.append(sector_edge_index)
                edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)
                pred = model(x, edge_index)
                mask = valid[d].to(device)
                target = lab[d].to(device)
                if mask.sum() < 10:
                    continue
                loss = F.mse_loss(pred[mask], target[mask])
                val_loss_sum += loss.item()
                val_count += 1
                val_preds[d] = pred.cpu().numpy()

        avg_val = val_loss_sum / max(val_count, 1)
        history['val_loss'].append(avg_val)

        # Per-epoch val IC
        ic_arr, _ = compute_daily_ic(val_preds, labels[horizon],
                                      label_valid[horizon], val_days)
        val_ic = ic_arr.mean() if len(ic_arr) > 0 else 0
        history['val_ic'].append(val_ic)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'  [seed={seed}] Epoch {epoch+1:3d}: train={avg_train:.5f} '
                  f'val={avg_val:.5f} val_IC={val_ic:.5f} lr={lr:.1e}')

        if no_improve >= patience:
            if verbose:
                print(f'  [seed={seed}] Early stop at epoch {epoch+1}')
            break

    # Load best and evaluate on test
    model.load_state_dict(best_state)
    model = model.to(device)
    model.eval()

    # Test predictions
    test_preds = np.zeros((num_days, num_stocks), dtype=np.float32)
    with torch.no_grad():
        for d in test_days:
            x = stock_features_t[d].to(device)
            edge_lists = []
            if 'corr' in edge_types:
                snap = corr_day_to_snapshot.get(d, 0)
                edge_lists.append(corr_snapshots[snap])
            if 'sector' in edge_types:
                edge_lists.append(sector_edge_index)
            edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)
            pred = model(x, edge_index)
            test_preds[d] = pred.cpu().numpy()

    metrics = compute_metrics(test_preds, horizon, test_days,
                               f'GAT 21d (seed={seed})')
    metrics['seed'] = seed
    metrics['best_epoch'] = len(history['train_loss']) - no_improve
    metrics['total_epochs'] = len(history['train_loss'])

    # Clean up GPU
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return metrics, history, test_preds


def train_lstm_ranking(horizon, train_days, val_days, test_days,
                        seq_len=21, seed=42, epochs=None, patience=None,
                        verbose=True):
    """
    Train LSTM baseline for ranking prediction.
    For each day d, uses features from [d-seq_len+1, ..., d] as input sequence.
    Returns: (metrics_dict, history_dict)
    """
    epochs = epochs or PARAMS['epochs']
    patience = patience or PARAMS['patience']

    set_seed(seed)

    model = RankingLSTM(
        in_channels=stock_features_t.shape[-1],
        hidden=PARAMS['hidden_channels'],
        num_layers=1,
        dropout=PARAMS['dropout'],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS['lr'],
                                 weight_decay=PARAMS['weight_decay'])

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_ic': []}

    lab = labels_t[horizon]
    valid = label_valid_t[horizon]

    def get_seq(d):
        """Get feature sequence ending at day d: (num_stocks, seq_len, feat_dim)."""
        start = max(0, d - seq_len + 1)
        seq = stock_features_t[start:d+1]  # (actual_len, stocks, feat)
        # Pad if needed
        if seq.shape[0] < seq_len:
            pad = torch.zeros(seq_len - seq.shape[0], seq.shape[1], seq.shape[2])
            seq = torch.cat([pad, seq], dim=0)
        return seq.permute(1, 0, 2)  # (stocks, seq_len, feat)

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0
        train_count = 0
        optimizer.zero_grad()

        day_order = train_days[np.random.permutation(len(train_days))]
        for step, d in enumerate(day_order):
            x_seq = get_seq(d).to(device)  # (stocks, seq_len, feat)
            pred = model(x_seq)
            mask = valid[d].to(device)
            target = lab[d].to(device)
            if mask.sum() < 10:
                continue
            loss = F.mse_loss(pred[mask], target[mask])
            loss.backward()
            train_loss_sum += loss.item()
            train_count += 1
            if (step + 1) % PARAMS['grad_accum'] == 0 or step == len(day_order) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        avg_train = train_loss_sum / max(train_count, 1)
        history['train_loss'].append(avg_train)

        # Validate
        model.eval()
        val_loss_sum = 0
        val_count = 0
        val_preds = np.zeros((num_days, num_stocks), dtype=np.float32)
        with torch.no_grad():
            for d in val_days:
                x_seq = get_seq(d).to(device)
                pred = model(x_seq)
                mask = valid[d].to(device)
                target = lab[d].to(device)
                if mask.sum() < 10:
                    continue
                loss = F.mse_loss(pred[mask], target[mask])
                val_loss_sum += loss.item()
                val_count += 1
                val_preds[d] = pred.cpu().numpy()

        avg_val = val_loss_sum / max(val_count, 1)
        history['val_loss'].append(avg_val)

        ic_arr, _ = compute_daily_ic(val_preds, labels[horizon],
                                      label_valid[horizon], val_days)
        val_ic = ic_arr.mean() if len(ic_arr) > 0 else 0
        history['val_ic'].append(val_ic)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch + 1) % 5 == 0:
            print(f'  LSTM Epoch {epoch+1:3d}: train={avg_train:.5f} '
                  f'val={avg_val:.5f} val_IC={val_ic:.5f}')

        if no_improve >= patience:
            if verbose:
                print(f'  LSTM Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model = model.to(device)
    model.eval()

    # Test predictions
    test_preds = np.zeros((num_days, num_stocks), dtype=np.float32)
    with torch.no_grad():
        for d in test_days:
            x_seq = get_seq(d).to(device)
            pred = model(x_seq)
            test_preds[d] = pred.cpu().numpy()

    metrics = compute_metrics(test_preds, horizon, test_days,
                               f'LSTM (seed={seed}, seq={seq_len})')

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return metrics, history


print('Enhanced training functions defined.')
print('  train_gat_with_diagnostics() — GAT with per-epoch IC tracking')
print('  train_lstm_ranking() — LSTM sequence baseline')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXPERIMENT 1.1: Multi-Seed GAT 21d
# Run GAT on 21d horizon with 5 different seeds to measure IC stability
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

SEEDS = [42, 123, 456, 789, 1024]
HORIZON = 21

print(f'=== Multi-Seed GAT {HORIZON}d Experiment ===')
print(f'Seeds: {SEEDS}')
print(f'Edge types: corr + sector')
print()

gat_results = []
gat_histories = []
gat_predictions = {}

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    metrics, history, preds = train_gat_with_diagnostics(
        horizon=HORIZON,
        train_days=train_days,
        val_days=val_days,
        test_days=test_days,
        edge_types=['corr', 'sector'],
        seed=seed,
        verbose=True,
    )
    gat_results.append(metrics)
    gat_histories.append(history)
    gat_predictions[seed] = preds
    print(f'  -> IC={metrics["IC"]:.5f}, ICIR={metrics["ICIR"]:.3f}, '
          f'Sharpe={metrics["Sharpe_LS"]:.3f}, '
          f'Best epoch={metrics["best_epoch"]}/{metrics["total_epochs"]}')

# Summary statistics
print(f'\n{"="*60}')
print(f'GAT {HORIZON}d Multi-Seed Summary ({len(SEEDS)} seeds)')
print(f'{"="*60}')

ics = [r['IC'] for r in gat_results]
icirs = [r['ICIR'] for r in gat_results]
sharpes = [r['Sharpe_LS'] for r in gat_results]

print(f'IC:     mean={np.mean(ics):.5f} ± {np.std(ics):.5f}  '
      f'(CV={np.std(ics)/abs(np.mean(ics))*100:.1f}%)')
print(f'ICIR:   mean={np.mean(icirs):.3f} ± {np.std(icirs):.3f}')
print(f'Sharpe: mean={np.mean(sharpes):.3f} ± {np.std(sharpes):.3f}')
print(f'IC range: [{min(ics):.5f}, {max(ics):.5f}]')

# Stability verdict
ic_mean = np.mean(ics)
ic_std = np.std(ics)
if ic_mean > 0.03 and ic_std < 0.02:
    print(f'\n✓ STABLE: mean IC > 0.03 and std < 0.02')
elif ic_mean > 0.03:
    print(f'\n⚠ UNSTABLE but mean IC > 0.03 (std={ic_std:.5f})')
    print(f'  Consider: ensemble averaging, switch to SAGE, or reduce lr')
else:
    print(f'\n✗ WEAK: mean IC < 0.03')
    print(f'  Consider: switch to SAGE (CV=2% in prior runs)')

# Per-seed table
print(f'\nPer-seed results:')
df_gat = pd.DataFrame(gat_results)
print(df_gat[['name', 'IC', 'ICIR', 'Sharpe_LS', 'Ann_LS_net',
              'best_epoch', 'total_epochs']].to_string(index=False))

# Save results
df_gat.to_csv('experiments/gat_21d_multiseed.csv', index=False)
print(f'\nResults saved to experiments/gat_21d_multiseed.csv')
print(f'Experiment 1.1 complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXPERIMENT 1.2: Multi-Seed LightGBM 21d (Control)
# Verify evaluation pipeline stability with deterministic-ish model
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

print(f'=== Multi-Seed LightGBM {HORIZON}d Control ===')

lgb_results = []
for seed in SEEDS:
    print(f'  Seed {seed}...', end=' ')
    res = train_lightgbm(
        horizon=HORIZON,
        train_days=train_days,
        val_days=val_days,
        test_days=test_days,
        seed=seed,
        name=f'LightGBM (seed={seed})',
    )
    if res:
        lgb_results.append(res)
        print(f'IC={res["IC"]:.5f}')

if lgb_results:
    lgb_ics = [r['IC'] for r in lgb_results]
    print(f'\nLightGBM {HORIZON}d: IC mean={np.mean(lgb_ics):.5f} ± {np.std(lgb_ics):.5f} '
          f'(CV={np.std(lgb_ics)/abs(np.mean(lgb_ics))*100:.1f}%)')

    df_lgb = pd.DataFrame(lgb_results)
    df_lgb.to_csv('experiments/lgb_21d_multiseed.csv', index=False)
    print('Results saved to experiments/lgb_21d_multiseed.csv')

print(f'\nExperiment 1.2 complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXPERIMENT 1.3: Training Diagnostics — Visualization
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'GAT {HORIZON}d Training Diagnostics ({len(SEEDS)} seeds)', fontsize=14)

colors = plt.cm.tab10(np.linspace(0, 1, len(SEEDS)))

# --- Panel 1: Val IC per epoch ---
ax = axes[0, 0]
for i, (seed, hist) in enumerate(zip(SEEDS, gat_histories)):
    ax.plot(hist['val_ic'], color=colors[i], alpha=0.7, label=f'seed={seed}')
ax.axhline(y=0.03, color='red', linestyle='--', alpha=0.5, label='IC=0.03 threshold')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation IC')
ax.set_title('Val IC per Epoch')
ax.legend(fontsize=8)

# --- Panel 2: Val Loss per epoch ---
ax = axes[0, 1]
for i, (seed, hist) in enumerate(zip(SEEDS, gat_histories)):
    ax.plot(hist['val_loss'], color=colors[i], alpha=0.7, label=f'seed={seed}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss (MSE)')
ax.set_title('Val Loss per Epoch')
ax.legend(fontsize=8)

# --- Panel 3: Train Loss per epoch ---
ax = axes[1, 0]
for i, (seed, hist) in enumerate(zip(SEEDS, gat_histories)):
    ax.plot(hist['train_loss'], color=colors[i], alpha=0.7, label=f'seed={seed}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss (MSE)')
ax.set_title('Train Loss per Epoch')
ax.legend(fontsize=8)

# --- Panel 4: Test IC Distribution ---
ax = axes[1, 1]
ics = [r['IC'] for r in gat_results]
ax.bar(range(len(SEEDS)), ics, color=colors[:len(SEEDS)], alpha=0.8)
ax.axhline(y=np.mean(ics), color='black', linestyle='--', label=f'Mean={np.mean(ics):.4f}')
ax.axhline(y=0.03, color='red', linestyle='--', alpha=0.5, label='IC=0.03')
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels([f's={s}' for s in SEEDS])
ax.set_ylabel('Test IC')
ax.set_title('Test IC per Seed')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/stability_gat_21d_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/stability_gat_21d_diagnostics.png')

# --- Ensemble analysis ---
print(f'\n=== Ensemble Analysis (average of {len(SEEDS)} seed predictions) ===')
ensemble_preds = np.mean([gat_predictions[s] for s in SEEDS], axis=0)
ensemble_metrics = compute_metrics(ensemble_preds, HORIZON, test_days,
                                    f'GAT {HORIZON}d Ensemble ({len(SEEDS)} seeds)')
print(f'Ensemble IC={ensemble_metrics["IC"]:.5f}, '
      f'ICIR={ensemble_metrics["ICIR"]:.3f}, '
      f'Sharpe={ensemble_metrics["Sharpe_LS"]:.3f}')
print(f'vs Individual mean IC={np.mean(ics):.5f}')
print(f'Ensemble improvement: {(ensemble_metrics["IC"] - np.mean(ics)):.5f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXPERIMENT 1.4: LSTM Baseline (no graph structure)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

print(f'=== LSTM Baseline ({HORIZON}d horizon, seq_len=21) ===')

lstm_metrics, lstm_history = train_lstm_ranking(
    horizon=HORIZON,
    train_days=train_days,
    val_days=val_days,
    test_days=test_days,
    seq_len=21,
    seed=42,
    verbose=True,
)

print(f'\nLSTM Results:')
print(f'  IC={lstm_metrics["IC"]:.5f}, ICIR={lstm_metrics["ICIR"]:.3f}, '
      f'Sharpe={lstm_metrics["Sharpe_LS"]:.3f}')
print(f'  Ann LS net: {lstm_metrics["Ann_LS_net"]}')

# Also run MLP baseline (GAT without edges = MLP)
print(f'\n=== MLP Baseline (no graph edges) ===')
set_seed(42)
mlp_model = RankingGNN(
    in_channels=stock_features_t.shape[-1],
    hidden=PARAMS['hidden_channels'],
    conv_type='gat',
    num_layers=PARAMS['num_hgt_layers'],
    dropout=PARAMS['dropout'],
)

# Train with empty edge_index (degenerates to MLP since no message passing)
mlp_metrics, mlp_history, _ = train_gat_with_diagnostics(
    horizon=HORIZON,
    train_days=train_days,
    val_days=val_days,
    test_days=test_days,
    edge_types=[],  # No edges = MLP
    seed=42,
    verbose=True,
)
print(f'\nMLP Results:')
print(f'  IC={mlp_metrics["IC"]:.5f}, ICIR={mlp_metrics["ICIR"]:.3f}, '
      f'Sharpe={mlp_metrics["Sharpe_LS"]:.3f}')

print(f'\nExperiment 1.4 complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SUMMARY — All Week 1 Results
# ══════════════════════════════════════════════════════════════════

print('=' * 70)
print('WEEK 1 STABILITY EXPERIMENTS — SUMMARY')
print('=' * 70)

# Collect all results
all_results = []

# GAT multi-seed summary
gat_ics = [r['IC'] for r in gat_results]
all_results.append({
    'Model': f'GAT 21d (mean of {len(SEEDS)} seeds)',
    'IC': f'{np.mean(gat_ics):.5f} ± {np.std(gat_ics):.5f}',
    'IC_CV': f'{np.std(gat_ics)/abs(np.mean(gat_ics))*100:.1f}%',
    'Sharpe': f'{np.mean([r["Sharpe_LS"] for r in gat_results]):.3f}',
})

# Ensemble
all_results.append({
    'Model': f'GAT 21d Ensemble ({len(SEEDS)} seeds)',
    'IC': f'{ensemble_metrics["IC"]:.5f}',
    'IC_CV': 'N/A',
    'Sharpe': f'{ensemble_metrics["Sharpe_LS"]:.3f}',
})

# LightGBM
if lgb_results:
    lgb_ics = [r['IC'] for r in lgb_results]
    all_results.append({
        'Model': f'LightGBM 21d (mean of {len(SEEDS)} seeds)',
        'IC': f'{np.mean(lgb_ics):.5f} ± {np.std(lgb_ics):.5f}',
        'IC_CV': f'{np.std(lgb_ics)/abs(np.mean(lgb_ics))*100:.1f}%',
        'Sharpe': f'{np.mean([r["Sharpe_LS"] for r in lgb_results]):.3f}',
    })

# LSTM
all_results.append({
    'Model': 'LSTM 21d (seq=21)',
    'IC': f'{lstm_metrics["IC"]:.5f}',
    'IC_CV': 'N/A (single run)',
    'Sharpe': f'{lstm_metrics["Sharpe_LS"]:.3f}',
})

# MLP
all_results.append({
    'Model': 'MLP 21d (no graph)',
    'IC': f'{mlp_metrics["IC"]:.5f}',
    'IC_CV': 'N/A (single run)',
    'Sharpe': f'{mlp_metrics["Sharpe_LS"]:.3f}',
})

df_summary = pd.DataFrame(all_results)
print(df_summary.to_string(index=False))

# Save comprehensive results
df_summary.to_csv('experiments/week1_summary.csv', index=False)
print(f'\nSaved: experiments/week1_summary.csv')

# Decision guidance
print(f'\n=== Decision Guidance ===')
gat_mean_ic = np.mean(gat_ics)
gat_std_ic = np.std(gat_ics)
if gat_mean_ic > 0.03 and gat_std_ic < 0.02:
    print('GAT 21d: STABLE and significant (mean IC > 0.03, std < 0.02)')
    print('-> Proceed to Week 2 (Walk-Forward CV)')
elif gat_mean_ic > 0.03:
    print(f'GAT 21d: Significant but UNSTABLE (IC CV = {gat_std_ic/gat_mean_ic*100:.0f}%)')
    print('-> Consider ensemble approach or switch primary model to SAGE')
    print(f'-> Ensemble IC = {ensemble_metrics["IC"]:.5f} (may stabilize)')
else:
    print(f'GAT 21d: NOT significant (mean IC = {gat_mean_ic:.5f} < 0.03)')
    print('-> Consider SAGE as primary model (CV=2% in prior runs)')
    print('-> Or investigate training hyperparameters (lr, dropout, grad_accum)')

## Next StepsBased on results above:1. If GAT stable → proceed to Walk-Forward CV (Week 2)2. If GAT unstable → try SAGE multi-seed, or use ensemble3. Upload results to `docs/analysis.md` with entry `2026-04-XX-a`4. Update `progress.md` and `plan.md`